# AutoDDG Multi-Provider Playground

This notebook shows how to experiment with the multi-LLM-provider helpers that ship with `AutoDDG`.
They wrap the shared configuration file so you can authenticate against different OpenAI-compatible
providers and generate dataset descriptions through the same interface as seen in `quick_start.ipynb`.


## Imports and sample dataset

We will reuse the Clark dataset sample that ships with the repository. Feel free to point the
helper to your own CSV sample if you prefer.

In [ ]:

import pandas as pd

from autoddg import AutoDDG

sample_df = pd.read_csv("clark_dataset.csv")
sample_df.head()

## Authenticate against a provider

Update the `provider_name` and `model_name` variables to match the platform and model you want to
exercise. If you prefer not to rely on environment variables, set `api_key_override` directly.
Any extra keyword arguments are forwarded to `litellm.completion` via `AutoDDG().with_provider(...)`.


In [ ]:
provider_name = "openai"  # e.g. "grok", "anthropic", "mistral"
model_name = "gpt-4o-mini"  # Replace with your target model identifier
api_key_override = None  # Set to a string to bypass environment variables

autoddg = AutoDDG().with_provider(
    provider=provider_name,
    model_name=model_name,
    api_key=api_key_override,
)

autoddg

## Generate a dataset description

The helper returns both the prompt that was sent to the LLM and the generated description. This
makes it easy to debug or reproduce results.

In [ ]:
dataset_sample = sample_df.head(10).to_csv(index=False)

prompt, description = autoddg.describe_dataset(dataset_sample=dataset_sample)
print(description)


## Compare providers side-by-side

You can wrap the previous logic in a simple loop to try multiple providers and models in a single
session. Uncomment or modify the entries below to match the credentials you have on hand.

In [ ]:
base = AutoDDG(description_words=100)

# Set up the various providers you want to try to generate description of your dataset from
# Please at least try two at the same time by uncomenting the lines below.
provider_trials = [
    # {"provider": "openai", "model": "gpt-4o-mini", "api_key": "sk-..."},
    # {"provider": "anthropic", "model": "claude-sonnet-4-5-20250929", "api_key": "sk-..."},
    # {"provider": "mistral", "model": "mistral-large-latest", "api_key": "..."},
    # {"provider": "grok", "model": "grok-3-mini-beta", "api_key": "..."},
]

dataset_sample = sample_df.head(10).to_csv(index=False)

comparisons = []
for trial in provider_trials:
    runner = base.with_provider(
        provider=trial["provider"],
        model_name=trial["model"],
        api_key=trial.get("api_key"),
    )
    _, output = runner.describe_dataset(dataset_sample=dataset_sample)
    comparisons.append({
        "provider": trial["provider"],
        "model": trial["model"],
        "description": output,
    })

comparison_df = pd.DataFrame(comparisons)
comparison_df


## Visualise description differences

Use the helpers below to explore similarities and differences between provider outputs.
The heatmap summarises pairwise similarity scores while the HTML tables highlight textual
diffs against a reference provider. For future work, explore SOTA NLP viz. libraries.


In [ ]:
from difflib import HtmlDiff, SequenceMatcher

from IPython.display import HTML, display


def similarity_heatmap(comparison_records):
    if not comparison_records:
        return None
    labels = [f"{row['provider']} · {row['model']}" for row in comparison_records]
    matrix = []
    for left in comparison_records:
        row = []
        for right in comparison_records:
            ratio = SequenceMatcher(None, left['description'], right['description']).ratio()
            row.append(ratio)
        matrix.append(row)
    frame = pd.DataFrame(matrix, index=labels, columns=labels)
    return frame.style.format('{:.2f}').background_gradient(cmap="Blues")

def display_diffs(comparison_records, reference_index=0, *, wrapcolumn=88):
    if not comparison_records:
        print("No comparison data available.")
        return
    reference = comparison_records[reference_index]
    diff = HtmlDiff(wrapcolumn=wrapcolumn)
    for _, entry in enumerate(comparison_records):
        table = diff.make_table(
            reference['description'].splitlines(),
            entry['description'].splitlines(),
            fromdesc=f"{reference['provider']} · {reference['model']}",
            todesc=f"{entry['provider']} · {entry['model']}",
            context=True,
            numlines=2,
        )
        display(HTML(table))


In [ ]:
heatmap = similarity_heatmap(comparisons)
heatmap

In [ ]:
display_diffs(comparisons)

## Provider helpers

`AutoDDG` exposes convenience helpers for introspecting the bundled provider configuration
and is backed by the same factory you can use in code. You can register additional providers
at runtime if needed, for that refer to the README.


In [ ]:
# List all configured provider identifiers
AutoDDG.list_providers()

In [ ]:
# Peek at one provider's configuration details
provider_overview = AutoDDG.describe_provider("openai")
provider_overview
